# Smart Energy Consumption Analytics

Pipeline for smart meter data ingestion, cleaning, anomaly detection, aggregations, and SMA forecasting.

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

readings_path = Path("../meter_readings.csv")
household_path = Path("../household_info.csv")

if not readings_path.exists():
    readings_path = Path("meter_readings.csv")
    household_path = Path("household_info.csv")

df_readings = pd.read_csv(readings_path)
df_households = pd.read_csv(household_path)

print(f"Loaded {len(df_readings)} meter readings across {len(df_households)} households.")
df_readings.head()

## 2. Cleansing & Anomaly Detection

In [ ]:
# Parse timestamps & sort
df_readings["timestamp"] = pd.to_datetime(df_readings["timestamp"])
df_readings = df_readings.sort_values(["meter_id", "timestamp"]).reset_index(drop=True)

# Merge household details
df_merged = df_readings.merge(df_households, on="household_id", how="left")

# Rule 1: Spike (reading >= 3x rolling mean)
df_merged["rolling_mean_3h"] = (
    df_merged.groupby("meter_id")["units_consumed"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    .fillna(df_merged["units_consumed"])
)
df_merged["is_spike"] = df_merged["units_consumed"] >= (3.0 * df_merged["rolling_mean_3h"])

# Rule 2: Extended Zero Outage (zero consumption >= 3 consecutive hours)
is_zero = df_merged["units_consumed"] == 0.0
zero_streak = is_zero.groupby((~is_zero).cumsum()).cumsum()
df_merged["is_zero_extended"] = zero_streak >= 3

# Rule 3: Pattern Deviation (outside Mean +- 2 std for hour of week)
df_merged["hour_of_week"] = df_merged["timestamp"].dt.dayofweek * 24 + df_merged["timestamp"].dt.hour
how_stats = df_merged.groupby(["meter_id", "hour_of_week"])["units_consumed"].agg(["mean", "std"]).reset_index()
df_merged = df_merged.merge(how_stats, on=["meter_id", "hour_of_week"], how="left")
df_merged["is_deviation"] = (
    (df_merged["std"] > 0) &
    ((df_merged["units_consumed"] > df_merged["mean"] + 2 * df_merged["std"]) |
     (df_merged["units_consumed"] < df_merged["mean"] - 2 * df_merged["std"]))
)

df_merged["is_anomaly"] = df_merged["is_spike"] | df_merged["is_zero_extended"] | df_merged["is_deviation"]

spikes_n = df_merged['is_spike'].sum()
zeros_n = df_merged['is_zero_extended'].sum()
devs_n = df_merged['is_deviation'].sum()
print(f"Total anomalies detected: {df_merged['is_anomaly'].sum()} (Spikes: {spikes_n}, Zeros: {zeros_n}, Deviations: {devs_n})")

## 3. Aggregations & Demand Forecasting (7-Day SMA)

In [ ]:
df_merged["date"] = df_merged["timestamp"].dt.date
df_daily = df_merged.groupby(["meter_id", "date", "city"])["units_consumed"].agg(["sum", "mean", "max"]).reset_index()

df_daily = df_daily.sort_values(["meter_id", "date"])
df_daily["sma_7d_forecast"] = (
    df_daily.groupby("meter_id")["mean"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

df_daily.head(10)

## 4. Visualization

In [ ]:
sample_meter = df_merged["meter_id"].iloc[0]
m_df = df_merged[df_merged["meter_id"] == sample_meter].sort_values("timestamp")

plt.figure(figsize=(14, 5))
plt.plot(m_df["timestamp"], m_df["units_consumed"], label="Consumption (kWh)", color="#1f77b4")

spikes = m_df[m_df["is_spike"]]
plt.scatter(spikes["timestamp"], spikes["units_consumed"], color="red", s=80, label="Spike Anomaly", marker="^")

zeros = m_df[m_df["is_zero_extended"]]
plt.scatter(zeros["timestamp"], zeros["units_consumed"], color="orange", s=60, label="Zero Outage", marker="o")

plt.title(f"Consumption & Anomalies for Meter {sample_meter}")
plt.xlabel("Timestamp")
plt.ylabel("kWh")
plt.legend()
plt.tight_layout()
plt.show()